In [0]:
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown(
    "schema",
    "migration_dev",
    ["migration_dev", "migration_test", "migration_prod_sim"],
)
dbutils.widgets.text("batch_id", "batch_001")

catalog_name = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog_name):
    raise ValueError("Invalid catalog name")

if schema_name not in {
    "migration_dev", "migration_test", "migration_prod_sim"
}:
    raise ValueError("Invalid environment schema")

if not re.fullmatch(r"batch_[0-9]+", batch_id):
    raise ValueError("Invalid batch ID")

table_prefix = f"{catalog_name}.{schema_name}"
volume_root = f"/Volumes/{catalog_name}/{schema_name}"

print(f"Environment: {table_prefix}")
print(f"Batch: {batch_id}")
print(f"Volume root: {volume_root}")    

In [0]:
from pyspark.sql import functions as F

controls = (
    spark.table(f"{table_prefix}.batch_control")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

if len(controls) != 1:
    raise RuntimeError("Expected one batch control record")

control = controls[0]

latest = (
    spark.table(f"{table_prefix}.downstream_reconciliation")
    .filter(F.col("batch_id") == batch_id)
    .orderBy(F.col("reconciled_at").desc())
    .limit(1)
    .collect()
)

if (
    control["batch_status"] != "APPROVED"
    or control["failed_checks"] != 0
    or not latest
    or latest[0]["load_completion_status"] != "COMPLETE"
    or latest[0]["validation_run_id"] != control["validation_run_id"]
):
    raise RuntimeError("Current batch is not approved and reconciled")

gold = (
    spark.table(f"{table_prefix}.gold_load_manifest")
    .filter(F.col("batch_id") == batch_id)
)

archive_files = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(f"{volume_root}/archive/{batch_id}")
    .select(
        F.regexp_extract("path", r"([^/]+)$", 1).alias("file_name"),
        F.regexp_replace("path", r"^dbfs:", "").alias("archive_path"),
        F.sha2("content", 256).alias("archive_sha256"),
        F.col("length").alias("archive_size"),
    )
)

documents = gold.join(archive_files, "file_name", "inner")

expected = control["expected_document_count"]

if (
    gold.count() != expected
    or archive_files.count() != expected
    or documents.count() != expected
    or documents.select("document_id").distinct().count() != expected
):
    raise RuntimeError("Archive count or document identity mismatch")

invalid = documents.filter(
    ~F.col("source_sha256").eqNullSafe(F.col("archive_sha256"))
    | ~F.col("file_size_bytes").eqNullSafe(F.col("archive_size"))
    | ~F.col("validation_run_id").eqNullSafe(control["validation_run_id"])
)

if invalid.limit(1).count() > 0:
    raise RuntimeError("Archive integrity or approval mismatch")

registry = documents.select(
    "batch_id",
    "document_id",
    "account_number",
    "document_type",
    "statement_date",
    "archive_path",
    F.col("archive_sha256").alias("document_sha256"),
    "file_size_bytes",
    "validation_run_id",
    F.lit(latest[0]["reconciliation_run_id"]).alias("reconciliation_run_id"),
    F.current_timestamp().alias("archived_at"),
    F.add_months(F.current_date(), 120).alias("retention_until"),
    F.lit(False).alias("legal_hold"),
)

registry.createOrReplaceTempView("current_registry_batch")

# Block conflicts with an existing registered document.
conflicts = spark.sql(f"""
    SELECT COUNT(*) AS conflicts
    FROM {table_prefix}.document_registry t
    JOIN current_registry_batch s
      ON t.batch_id = s.batch_id AND t.document_id = s.document_id
    WHERE NOT (t.document_sha256 <=> s.document_sha256)
       OR NOT (t.archive_path <=> s.archive_path)
       OR NOT (t.account_number <=> s.account_number)
       OR NOT (t.document_type <=> s.document_type)
       OR NOT (t.statement_date <=> s.statement_date)
       OR NOT (t.file_size_bytes <=> s.file_size_bytes)
""").first()["conflicts"]

if conflicts:
    raise RuntimeError("Existing registry records conflict with this batch")

# Preserve existing records, including retention dates and legal holds.
spark.sql(f"""
    MERGE INTO {table_prefix}.document_registry AS target
    USING current_registry_batch AS source
    ON target.batch_id = source.batch_id
       AND target.document_id = source.document_id
    WHEN NOT MATCHED THEN INSERT *
""")

saved = (
    spark.table(f"{table_prefix}.document_registry")
    .filter(F.col("batch_id") == batch_id)
)

print(f"Registered documents: {saved.count()}")
display(saved.orderBy("document_id").limit(5))